# Do Full-Duplex Speech Models Listen in Text? — v8: поведенческий тест «голосового шлюза»

v7 показал: голос в собственном канале модели подавляет **текстовое представление** собеседника, но это представление почти не используется для ответа. Значит, нужно прямо проверить, теряет ли **ответ** информацию об услышанном, когда модель говорит одновременно с собеседником.

**Дизайн.** Для каждой реплики и каждого варианта собственного канала (как в v7) делаем два прогона: собеседник говорит / собеседник заглушён. Через 0,5 с после реплики в собственный текст модели ставится EPAD; распределение первого слова ответа сравнивается:
- **влияние собеседника** `U(c) = KL(p[c, собеседник есть] ‖ p[c, собеседник заглушён])`;
- **вероятность услышанных слов** `Δlog P(c) = log P_heard[c, есть] − log P_heard[c, заглушён]`.

**Сохранность информации** `retention(c) = U(c) / U(listen)` (и то же для `Δlog P`): 1 — ответ зависит от слов собеседника так же, как когда модель молча слушала; 0 — ответ не зависит от них вовсе. Сравнение с заглушённым собеседником внутри того же условия убирает влияние самой собственной речи на ответ.

Выравнивание слов не нужно (используется эталонный текст), прогон ~30–45 мин на модель. Kaggle: GPU T4×2, Internet On, `HF_TOKEN`.

## 1. Конфигурация

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
BASE = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.getcwd())

GAINS_DB = [-30, -10]
CFG = dict(
    model          = "personaplex",              # "personaplex" | "moshi"
    datasets       = ["librispeech", "ami"],
    conditions     = ["listen", "talking_audio", "own_reversed", "own_noise", "own_echo", "own_gain-30", "own_gain-10"],
    echo_delay_sec = 0.2,
    echo_gain_db   = -10,
    n_turns        = 60,
    turn_sec       = 20.0,
    gap_sec        = 0.3,
    silence_frames = 6,
    post_sec       = 1.5,
    resp_gap_sec   = 0.5,
    min_word_chars = 4,
    n_boot         = 2000,
    tl_extra       = [],                          # no tuned-lens data needed here
    tl_extra_turns = 0,
    dtype          = "auto",
    seed           = 0,
)
PRESETS = {
    "personaplex": dict(model_repo="nvidia/personaplex-7b-v1",
                        text_prompt="You enjoy having a good conversation.", prefix_user="sine"),
    "moshi":       dict(model_repo="kyutai/moshiko-pytorch-bf16", text_prompt="", prefix_user="silence"),
}
CFG.update(PRESETS[CFG["model"]])
CFG["out_dir"] = f"{BASE}/{CFG['model']}_v8"
os.makedirs(CFG["out_dir"], exist_ok=True)
print(CFG["model_repo"], "->", CFG["out_dir"])

## 2. Установка, модель, данные

In [ ]:
import sys, subprocess
os.environ["NO_TORCH_COMPILE"] = "1"
os.environ["NO_CUDA_GRAPH"] = "1"
if not os.path.exists("personaplex"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/NVIDIA/personaplex.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "sentencepiece", "sphn", "openai-whisper",
                "soundfile", "tqdm", "wordfreq", "datasets"], check=True)
sys.path.insert(0, os.path.abspath("personaplex/moshi"))

In [ ]:
import json, math, random, re, tarfile, urllib.request, gc, io, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import sentencepiece, soundfile as sf
from scipy.signal import resample_poly
from safetensors import safe_open
from moshi.models import loaders
from moshi.models.lm import LMModel, SILENCE_TOKENS, SINE_TOKENS, _delay_sequence
warnings.filterwarnings("ignore", category=RuntimeWarning)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
if not os.environ.get("HF_TOKEN"):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        elif ON_COLAB:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("could not read HF_TOKEN secret:", e)
repo = CFG["model_repo"]
files = list_repo_files(repo)
lm_file = loaders.MOSHI_NAME if loaders.MOSHI_NAME in files else next(
    f for f in files if f.endswith(".safetensors") and not f.startswith("tokenizer"))
moshi_path = hf_hub_download(repo, lm_file)
mimi_path  = hf_hub_download(repo, loaders.MIMI_NAME)
sp = sentencepiece.SentencePieceProcessor(hf_hub_download(repo, loaders.TEXT_TOKENIZER_NAME))

In [ ]:
SPECIAL_TEXT = [0, 1, 2, 3]   # EPAD, BOS, EOS, PAD
SKIP_PREFIXES = ("depformer", "linears.")

def pick_dtype():
    if CFG["dtype"] != "auto":
        return {"fp16": torch.float16, "bf16": torch.bfloat16}[CFG["dtype"]]
    major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
    return torch.bfloat16 if major >= 8 else torch.float16

def layer_devices(n_layers):
    n_gpu = torch.cuda.device_count()
    if n_gpu == 0: return ["cpu"] * n_layers
    if n_gpu == 1: return ["cuda:0"] * n_layers
    half = n_layers // 2
    return ["cuda:0"] * half + ["cuda:1"] * (n_layers - half)

def load_lm_for_lens(path, lm_kwargs, dtype):
    with torch.device("meta"):
        lm = LMModel(device="meta", dtype=dtype, **dict(lm_kwargs))
    devs = layer_devices(len(lm.transformer.layers))
    expected = set(lm.state_dict().keys())
    def target_device(key):
        m = re.match(r"transformer\.layers\.(\d+)\.", key)
        if m: return devs[int(m.group(1))]
        if key.startswith(("emb.", "text_emb.")): return devs[0]
        return devs[-1]
    sd = {}
    with safe_open(path, framework="pt", device="cpu") as f:
        for key in f.keys():
            if key.startswith(SKIP_PREFIXES) or key not in expected:
                continue
            t = f.get_tensor(key)
            sd[key] = t.to(device=target_device(key), dtype=torch.float32 if key.endswith(".alpha") else dtype)
    missing = [k for k in expected if k not in sd and not k.startswith(SKIP_PREFIXES)]
    assert not missing, f"missing weights: {missing[:5]}"
    lm.load_state_dict(sd, strict=False, assign=True)
    lm.eval()
    for p in lm.parameters():
        p.requires_grad_(False)
    return lm, devs

DTYPE = pick_dtype()
lm, LAYER_DEVS = load_lm_for_lens(moshi_path, dict(loaders._lm_kwargs, dep_q=16), DTYPE)
N_LAYERS = len(lm.transformer.layers) + 1
mimi = loaders.get_mimi(mimi_path, device="cuda:0" if torch.cuda.is_available() else "cpu")
FRAME = int(mimi.sample_rate / mimi.frame_rate); FPS = mimi.frame_rate
print("dtype", DTYPE, "| devices", sorted(set(LAYER_DEVS)))

In [ ]:
def norm_words(text):
    return [w for w in (re.sub(r"[^a-z']", "", t) for t in text.lower().split()) if w.strip("'")]

def librispeech_turns(n_turns, turn_sec, gap_sec, sr_out=24000, subset="test-clean"):
    root = Path(f"{BASE}/librispeech")
    if not (root / "LibriSpeech" / subset).exists():
        root.mkdir(parents=True, exist_ok=True)
        tgz = root / f"{subset}.tar.gz"
        urllib.request.urlretrieve(f"https://www.openslr.org/resources/12/{subset}.tar.gz", tgz)
        with tarfile.open(tgz) as t:
            t.extractall(root)
        tgz.unlink()
    chapters = sorted((root / "LibriSpeech" / subset).glob("*/*"))
    random.Random(CFG["seed"]).shuffle(chapters)
    turns = []
    for ch in chapters:
        trans = dict(l.split(" ", 1) for l in next(ch.glob("*.trans.txt")).read_text().splitlines())
        pieces, segs, t = [], [], 0.0
        for uid in sorted(trans):
            wav, sr = sf.read(ch / f"{uid}.flac", dtype="float32")
            wav = resample_poly(wav, sr_out // 1000, sr // 1000).astype(np.float32)
            segs.append((t, t + len(wav) / sr_out, trans[uid]))
            pieces += [wav, np.zeros(int(gap_sec * sr_out), np.float32)]
            t += len(wav) / sr_out + gap_sec
            if t >= turn_sec:
                break
        turns.append(dict(id=f"ls-{subset}-{ch.parent.name}-{ch.name}", spk=f"ls-{ch.parent.name}", wav=np.concatenate(pieces), segments=segs))
        if len(turns) >= n_turns:
            break
    return turns

def ami_turns(n_turns, turn_sec, gap_sec, sr_out=24000, max_items=20000):
    from datasets import load_dataset, Audio
    ds = load_dataset("edinburghcstr/ami", "ihm", split="test", streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False))
    by_spk = {}
    for k, ex in enumerate(ds):
        if k >= max_items: break
        key = (ex["meeting_id"], ex["speaker_id"])
        by_spk.setdefault(key, []).append(ex)
    turns = []
    for (meeting, spk), exs in sorted(by_spk.items()):
        exs.sort(key=lambda e: e["begin_time"])
        pieces, segs, t = [], [], 0.0
        for ex in exs:
            if len(norm_words(ex["text"])) < 3:
                continue
            a = ex["audio"]
            wav, sr = sf.read(io.BytesIO(a["bytes"]) if a.get("bytes") else a["path"], dtype="float32")
            if wav.ndim > 1: wav = wav.mean(1)
            wav = resample_poly(wav, sr_out, sr).astype(np.float32)
            segs.append((t, t + len(wav) / sr_out, ex["text"]))
            pieces += [wav, np.zeros(int(gap_sec * sr_out), np.float32)]
            t += len(wav) / sr_out + gap_sec
            if t >= turn_sec:
                turns.append(dict(id=f"ami-{meeting}-{spk}-{len(turns)}", spk=f"ami-{spk}", wav=np.concatenate(pieces), segments=segs))
                pieces, segs, t = [], [], 0.0
                if sum(1 for x in turns if x["spk"] == f"ami-{spk}") >= 3:
                    break                                    # at most 3 turns per speaker for diversity
        if len(turns) >= n_turns:
            break
    return turns

DATA = {}
for name in CFG["datasets"]:
    try:
        DATA[name] = (librispeech_turns if name == "librispeech" else ami_turns)(CFG["n_turns"], CFG["turn_sec"], CFG["gap_sec"])
        print(f"{name}: {len(DATA[name])} turns, {sum(len(t['wav']) for t in DATA[name]) / 24000 / 60:.1f} min, "
              f"{len({t['spk'] for t in DATA[name]})} speakers")
    except Exception as e:
        print(f"!! {name} skipped: {type(e).__name__}: {e}")
assert DATA, "no dataset loaded"
MAIN = [n for n in CFG["datasets"] if n in DATA]
for sub in CFG["tl_extra"]:
    try:
        DATA[f"ls_{sub}"] = librispeech_turns(CFG["tl_extra_turns"], CFG["turn_sec"], CFG["gap_sec"], subset=sub)
        print(f"ls_{sub}: {len(DATA[f'ls_{sub}'])} turns (tuned lens only)")
    except Exception as e:
        print(f"!! ls_{sub} skipped: {type(e).__name__}: {e}")

## 3. Варианты собственного канала (как в v7)

In [ ]:
@torch.no_grad()
def mimi_codes(wav):
    pad = (-len(wav)) % FRAME
    x = torch.from_numpy(np.pad(wav, (0, pad))).to(next(mimi.parameters()).device).view(1, 1, -1)
    return mimi.encode(x)[0].cpu()

def speech_shaped_noise(wav, seed):
    X = np.fft.rfft(wav)
    ph = np.random.default_rng(seed).uniform(0, 2 * np.pi, len(X))
    y = np.fft.irfft(np.abs(X) * np.exp(1j * ph), n=len(wav)).astype(np.float32)
    return y * (np.sqrt(np.mean(wav ** 2)) / (np.sqrt(np.mean(y ** 2)) + 1e-9))

def wrap_system(text):
    return f"<system> {text.strip()} <system>"

SIL = torch.as_tensor(SILENCE_TOKENS).view(8, 1)
OWN_SRC = {"talking_audio": "codes_user", "own_reversed": "codes_rev", "own_noise": "codes_noise"}
OWN_SRC.update({f"own_gain{g}": f"codes_gain{g}" for g in GAINS_DB})

def own_streams(other, n, cond, self_=None):
    codes = SIL.repeat(1, n).clone()
    text = np.full(n, 3, dtype=np.int64)
    if cond == "listen":
        return codes, text, (0, 0)
    src = self_[ "codes_echo"] if cond == "own_echo" else other[OWN_SRC[cond]]
    L = min(n, src.shape[1])
    codes[:, :L] = src[:, :L]
    return codes, text, (0, L)

def build_codes(user_codes, own_codes, own_text, n_post):
    pad = 3
    filler = torch.as_tensor(SINE_TOKENS).view(8, 1) if CFG["prefix_user"] == "sine" else SIL
    cols = []
    def frame(tt, ma, ua): cols.append(torch.cat([torch.tensor([[int(tt)]]), ma, ua], 0))
    for _ in range(CFG["silence_frames"]): frame(pad, SIL, filler)
    for tok in PROMPT_IDS: frame(tok, SIL, filler)
    for _ in range(CFG["silence_frames"]): frame(pad, SIL, filler)
    start = len(cols)
    for i in range(user_codes.shape[1]):
        frame(own_text[i], own_codes[:, i:i + 1], user_codes[:, i:i + 1])
    for _ in range(n_post): frame(pad, SIL, SIL)
    return torch.cat(cols, 1).long(), start

PROMPT_IDS = sp.encode(wrap_system(CFG["text_prompt"])) if CFG["text_prompt"] else []
N_POST = int(round(CFG["post_sec"] * FPS))
for name, turns in DATA.items():
    for i, t in enumerate(turns):
        t["codes_user"] = mimi_codes(t["wav"])
        if name in MAIN:
            t["codes_rev"] = mimi_codes(np.ascontiguousarray(t["wav"][::-1]))
            t["codes_noise"] = mimi_codes(speech_shaped_noise(t["wav"], CFG["seed"] + i))
            for gdb in GAINS_DB:
                t[f"codes_gain{gdb}"] = mimi_codes(t["wav"] * 10 ** (gdb / 20))
            d = int(CFG["echo_delay_sec"] * 24000)
            echo = np.concatenate([np.zeros(d, np.float32), t["wav"][:-d]]) * 10 ** (CFG["echo_gain_db"] / 20)
            t["codes_echo"] = mimi_codes(echo.astype(np.float32))
print("ok, post frames:", N_POST)

## 4. Вынужденный ответ: собеседник есть / заглушён

In [ ]:
SPECIAL_T = None
@torch.no_grad()
def first_word(codes, final_pos):
    first, last = LAYER_DEVS[0], LAYER_DEVS[-1]
    c = codes.unsqueeze(0).to(first)
    initial = lm._get_initial_token().to(first)
    seq = torch.cat([initial, _delay_sequence(lm.delays, c, initial)], 2)[:, :, :-1]
    x = lm.embed_codes(seq)
    for i, layer in enumerate(lm.transformer.layers):
        x = layer(x.to(LAYER_DEVS[i]))
    if not torch.isfinite(x).all():
        raise FloatingPointError("non-finite activations")
    return torch.log_softmax(lm.text_linear(lm.out_norm(x.to(last))).float()[0, final_pos], -1).cpu()

def turn_ids(t):
    ids = set()
    for _, _, text in t["segments"]:
        for w in norm_words(text):
            if len(w) >= CFG["min_word_chars"]:
                enc = sp.encode(w)
                if enc and enc[0] not in (0, 1, 2, 3):
                    ids.add(enc[0])
    return ids

GAP = int(round(CFG["resp_gap_sec"] * FPS))
rows = []
t0 = time.time()
for name in MAIN:
    turns = DATA[name]
    ids_all = [turn_ids(t) for t in turns]
    for ti, t in enumerate(turns):
        heard = torch.tensor(sorted(ids_all[ti]))
        if len(heard) < 3: continue
        other = turns[(ti + 1) % len(turns)]
        n = t["codes_user"].shape[1]
        silent_user = SIL.repeat(1, n)
        for cond in CFG["conditions"]:
            own_codes, own_text, _ = own_streams(other, n, cond, self_=t)
            out = {}
            for user_on in (True, False):
                codes, start = build_codes(t["codes_user"] if user_on else silent_user, own_codes, own_text, N_POST)
                b = start + n; e = b + GAP
                codes[0, e] = 0                                     # forced EPAD: the answer starts on the next frame
                lp = first_word(codes, list(range(b, e + 2)))
                out[user_on] = lp
            on, off = out[True], out[False]
            rows.append(dict(dataset=name, ti=ti, cond=cond,
                             kl=torch.sum(on[-1].exp() * (on[-1] - off[-1])).item(),
                             logp_on=float(torch.logsumexp(on[-1][heard], 0)), logp_off=float(torch.logsumexp(off[-1][heard], 0)),
                             p_epad_on=on[:-1, 0].exp().mean().item(), p_epad_off=off[:-1, 0].exp().mean().item(),
                             top1_on=int(on[-1].argmax()), top1_off=int(off[-1].argmax())))
    print(f"{name}: done ({time.time() - t0:.0f}s)")
R = pd.DataFrame(rows); R.to_csv(f"{CFG['out_dir']}/gate_behaviour_rows.csv", index=False)

## 5. Сохранность информации об услышанном

In [ ]:
boot = np.random.default_rng(CFG["seed"])
R["dlogp"] = R.logp_on - R.logp_off
res = []
for name, g in R.groupby("dataset"):
    base = g[g.cond == "listen"].set_index("ti")
    for cond, h in g.groupby("cond"):
        h = h.set_index("ti").loc[base.index]
        for metric in ("kl", "dlogp"):
            num, den = h[metric].values, base[metric].values
            idx = boot.integers(0, len(num), (CFG["n_boot"], len(num)))
            ratios = num[idx].mean(1) / den[idx].mean(1)
            res.append(dict(dataset=name, cond=cond, metric=metric, value=num.mean(), listen=den.mean(),
                            retention=num.mean() / den.mean(), lo=np.percentile(ratios, 2.5), hi=np.percentile(ratios, 97.5), n=len(num)))
        res.append(dict(dataset=name, cond=cond, metric="p_epad_on", value=h.p_epad_on.mean(), listen=base.p_epad_on.mean(),
                        retention=np.nan, lo=np.nan, hi=np.nan, n=len(h)))
RES = pd.DataFrame(res); RES.to_csv(f"{CFG['out_dir']}/gate_behaviour.csv", index=False)
pd.set_option("display.width", 200)
for name in MAIN:
    d = RES[(RES.dataset == name) & RES.metric.isin(["kl", "dlogp"])]
    print(f"\n=== {name}: information about the user's words retained in the answer (1 = as when listening) ===")
    print(d.pivot_table(index="cond", columns="metric", values=["value", "retention", "lo", "hi"]).round(3).to_string())
    e = RES[(RES.dataset == name) & (RES.metric == "p_epad_on")].set_index("cond").value
    print("P(EPAD) before the forced onset (readiness to speak):", e.round(4).to_dict())

order = [c for c in CFG["conditions"]]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, metric, lab in zip(axes, ("kl", "dlogp"), ("KL(user on ‖ user silenced)", "Δ log P(heard words)")):
    for k, name in enumerate(MAIN):
        d = RES[(RES.dataset == name) & (RES.metric == metric)].set_index("cond").loc[order]
        x = np.arange(len(order)) + (k - 0.5) * 0.35
        ax.bar(x, d.retention, 0.35, yerr=[d.retention - d.lo, d.hi - d.retention], capsize=3, label=name)
    ax.axhline(1, color="k", lw=.8, ls=":"); ax.axhline(0, color="k", lw=.8)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=20)
    ax.set(title=f"{CFG['model']}: retention of user information in the answer ({lab})", ylabel="relative to listen")
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/gate_behaviour.png", dpi=150); plt.show()

## 6. Сохранение

In [ ]:
json.dump(CFG, open(f"{CFG['out_dir']}/config.json", "w"), indent=2)
import shutil
from IPython.display import FileLink, display
archive = shutil.make_archive(CFG["out_dir"], "zip", CFG["out_dir"])
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")
os.chdir(BASE); display(FileLink(os.path.basename(archive)))

## Как читать
- `retention` ≈ 1 для `own_*`: модель продолжает учитывать слова собеседника, хотя их текстовое представление подавлено → «слышит, но не транскрибирует».
- `retention` заметно < 1 для речи и обращённой речи, но ≈ 1 для шума и эха (как в v7) → голосовой шлюз имеет поведенческие последствия.